# Common setup

Run these cells first.

In [1]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
from collections import Counter

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name == "vision" and PROJECT_ROOT.parent.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

MANIFEST_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/manifests"
RAW_VIDEO_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/raw_videos"
CLIP_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/clips_5s"
MODEL_DIR = PROJECT_ROOT / "storage/vision/models"
SAMPLE_MANIFEST = MANIFEST_DIR / "sample_700_coarse_manifest.csv"
DOWNLOAD_MANIFEST = MANIFEST_DIR / "train_700_download_manifest.csv"
CLIP_MANIFEST = MANIFEST_DIR / "train_700_clip_manifest_5s.csv"

PER_LABEL = 700
SEED = 42
DEVICE = "auto"
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /workspace/SKN27-FINAL-3Team


In [5]:
def run_command(command, *, timeout=None):
    print("$", " ".join(map(str, command)))
    completed = subprocess.run(list(map(str, command)), cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=timeout)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed


## Install and environment check

In [6]:
run_command([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], timeout=3600)
usage = shutil.disk_usage(PROJECT_ROOT)
print("free_gb:", round(usage.free / 1024**3, 2))
run_command([sys.executable, "-c", "import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"])


$ /usr/bin/python -m pip install -r requirements.txt


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip

free_gb: 117365.35
$ /usr/bin/python -c import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
cuda_available: True
device: NVIDIA RTX A5000



CompletedProcess(args=['/usr/bin/python', '-c', "import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"], returncode=0, stdout='cuda_available: True\ndevice: NVIDIA RTX A5000\n', stderr='')

## Build or check downloaded-video manifest

In [7]:
if not SAMPLE_MANIFEST.exists():
    raise FileNotFoundError(f"sample manifest not found: {SAMPLE_MANIFEST}")

if not DOWNLOAD_MANIFEST.exists():
    run_command([
        sys.executable,
        "etl/vision/download_sampled_media.py",
        "--input", SAMPLE_MANIFEST,
        "--output", DOWNLOAD_MANIFEST,
        "--download-dir", RAW_VIDEO_DIR,
        "--label-column", "coarse_label",
        "--per-label", str(PER_LABEL),
        "--split", "",
    ], timeout=None)

with DOWNLOAD_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    download_rows = list(csv.DictReader(f))
print("download_rows:", len(download_rows))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in download_rows)))
print("download_status_counts:", dict(Counter(row.get("download_status") for row in download_rows)))


$ /usr/bin/python etl/vision/download_sampled_media.py --input /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/sample_700_coarse_manifest.csv --output /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_download_manifest.csv --download-dir /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/raw_videos --label-column coarse_label --per-label 700 --split 
[1/2800]   0.0% exists: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/raw_videos/차대보행자/aihub_train_00000536_bb_1_190728_pedestrian_120_238.mp4
[2/2800]   0.1% download: aihub_train_00000181 차대보행자 -> /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/raw_videos/차대보행자/aihub_train_00000181_bb_1_200219_pedestrian_120_172.mp4
[2/2800]   0.1% done: aihub_train_00000181_bb_1_200219_pedestrian_120_172.mp4
[3/2800]   0.1% download: aihub_train_00000043 차대보행자 -> /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classificatio

# Model 1: YOLO/ByteTrack clip candidates + VideoMAE

Build total 5-second clips around an accident candidate, then train VideoMAE on those clips.

In [8]:
ACCIDENT_SOURCE = "yolo_track"  # use "center" if ByteTrack is too slow for all videos
YOLO_MODEL = "yolov8n.pt"

run_command([
    sys.executable,
    "etl/vision/build_training_clips.py",
    "--input", DOWNLOAD_MANIFEST,
    "--output", CLIP_MANIFEST,
    "--clip-dir", CLIP_DIR,
    "--label-column", "coarse_label",
    "--clip-sec", "5",
    "--accident-source", ACCIDENT_SOURCE,
    "--model-name", YOLO_MODEL,
    "--overwrite",
], timeout=None)

with CLIP_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    clip_rows = list(csv.DictReader(f))
print("clip_rows:", len(clip_rows))
print("clip_status_counts:", dict(Counter(row.get("clip_status") for row in clip_rows)))
print("clip_basis_counts:", dict(Counter(row.get("clip_basis") for row in clip_rows)))


$ /usr/bin/python etl/vision/build_training_clips.py --input /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_download_manifest.csv --output /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv --clip-dir /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/clips_5s --label-column coarse_label --clip-sec 5 --accident-source yolo_track --model-name yolov8n.pt --overwrite
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ok: aihub_train_00000536 0.97~5.97s basis=yolo_bytetrack_bbox_change
ok: aihub_train_00000181 0.00~5.00s basis=yolo_bytetrack_bbox_change
ok: aihub_train_00000043 5.00~10.00s basis=yolo_bytetrack_bbox_chang

## Shared VideoMAE helpers

In [10]:
VIDEOMAE_MODEL_DIR = MODEL_DIR / "videomae_classification_clip5s"
EARLY_STOPPING_PATIENCE = 2

def build_videomae_command(experiment):
    command = [
        sys.executable,
        "ai/vision/train_videomae_classifier.py",
        "--manifest", CLIP_MANIFEST,
        "--root-dir", PROJECT_ROOT,
        "--output-dir", VIDEOMAE_MODEL_DIR,
        "--label-column", "coarse_label",
        "--frame-count", str(experiment["frame_count"]),
        "--epochs", str(experiment["epochs"]),
        "--batch-size", str(experiment["batch_size"]),
        "--learning-rate", str(experiment["learning_rate"]),
        "--weight-decay", str(experiment["weight_decay"]),
        "--early-stopping-patience", str(EARLY_STOPPING_PATIENCE),
        "--seed", str(SEED),
        "--device", DEVICE,
        "--num-workers", "0",
        "--no-show-progress",
    ]
    if experiment["freeze_backbone"]:
        command.append("--freeze-backbone")
    return command

def latest_run_dir(output_dir):
    runs = [path for path in output_dir.iterdir() if path.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No run directories found: {output_dir}")
    return max(runs, key=lambda path: path.stat().st_mtime)

def show_run_result(run_dir):
    print("run_dir:", run_dir)
    for name in ["run_config.json", "training_history.csv"]:
        path = run_dir / name
        print("##", name, path.exists())
        if path.suffix == ".json" and path.exists():
            data = json.loads(path.read_text(encoding="utf-8"))
            for key in ["run_id", "freeze_backbone", "epochs", "batch_size", "learning_rate", "weight_decay", "best_epoch", "best_val_accuracy", "train_rows", "val_rows", "test_rows"]:
                if key in data:
                    print(key, data[key])
        elif path.exists():
            rows = list(csv.DictReader(path.open("r", encoding="utf-8")))
            for row in rows:
                print(row)
            if rows:
                print("best_val:", max(rows, key=lambda row: float(row.get("val_accuracy") or 0)))
                print("best_test:", max(rows, key=lambda row: float(row.get("test_accuracy") or 0)))


## Combination 1 - freeze baseline - define

In [11]:
EXPERIMENT = {'name': 'videomae_clip5s_baseline_freeze_lr1e-3_e5', 'epochs': 5, 'batch_size': 2, 'learning_rate': 0.001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': True}
print(EXPERIMENT)


{'name': 'videomae_clip5s_baseline_freeze_lr1e-3_e5', 'epochs': 5, 'batch_size': 2, 'learning_rate': 0.001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': True}


## Combination 1 - freeze baseline - train

In [12]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


$ /usr/bin/python ai/vision/train_videomae_classifier.py --manifest /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv --root-dir /workspace/SKN27-FINAL-3Team --output-dir /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s --label-column coarse_label --frame-count 16 --epochs 5 --batch-size 2 --learning-rate 0.001 --weight-decay 0.0 --early-stopping-patience 2 --seed 42 --device auto --num-workers 0 --no-show-progress --freeze-backbone
run_id: videomae_cls_20260702_124642
device: cuda
manifest: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv
labels: {'차대보행자': 0, '차대이륜차': 1, '차대자전거': 2, '차대차': 3}
rows: train=1773 val=420 test=212
epoch=1 train_loss=1.257951 train_acc=0.411168 val_loss=1.109658 val_acc=0.495238
epoch=2 train_loss=1.092203 train_acc=0.533559 val_loss=1.062401 val_acc=0.538095
epoch=3 train_loss=1.021974 train_acc=0.564580 va

## Combination 1 - freeze baseline - result

In [13]:
show_run_result(LAST_RUN_DIR)


run_dir: /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s/videomae_cls_20260702_124642
## run_config.json True
run_id videomae_cls_20260702_124642
freeze_backbone True
epochs 5
batch_size 2
learning_rate 0.001
weight_decay 0.0
best_epoch 2
best_val_accuracy 0.5380952380952381
train_rows 1773
val_rows 420
test_rows 212
## training_history.csv True
{'epoch': '1', 'train_loss': '1.257951', 'train_accuracy': '0.411168', 'val_loss': '1.109658', 'val_accuracy': '0.495238', 'test_loss': '1.154184', 'test_accuracy': '0.504717'}
{'epoch': '2', 'train_loss': '1.092203', 'train_accuracy': '0.533559', 'val_loss': '1.062401', 'val_accuracy': '0.538095', 'test_loss': '1.123605', 'test_accuracy': '0.547170'}
{'epoch': '3', 'train_loss': '1.021974', 'train_accuracy': '0.564580', 'val_loss': '1.041915', 'val_accuracy': '0.514286', 'test_loss': '1.075777', 'test_accuracy': '0.518868'}
{'epoch': '4', 'train_loss': '0.961263', 'train_accuracy': '0.600113', 'val_loss': '1.0

## Combination 2 - unfreeze lr 1e-4 - define

In [14]:
EXPERIMENT = {'name': 'videomae_clip5s_exp2_lr1e-4_e10', 'epochs': 10, 'batch_size': 2, 'learning_rate': 0.0001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


{'name': 'videomae_clip5s_exp2_lr1e-4_e10', 'epochs': 10, 'batch_size': 2, 'learning_rate': 0.0001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': False}


## Combination 2 - unfreeze lr 1e-4 - train

In [15]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


$ /usr/bin/python ai/vision/train_videomae_classifier.py --manifest /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv --root-dir /workspace/SKN27-FINAL-3Team --output-dir /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s --label-column coarse_label --frame-count 16 --epochs 10 --batch-size 2 --learning-rate 0.0001 --weight-decay 0.0 --early-stopping-patience 2 --seed 42 --device auto --num-workers 0 --no-show-progress
run_id: videomae_cls_20260702_153544
device: cuda
manifest: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv
labels: {'차대보행자': 0, '차대이륜차': 1, '차대자전거': 2, '차대차': 3}
rows: train=1773 val=420 test=212
epoch=1 train_loss=1.388905 train_acc=0.279188 val_loss=1.265986 val_acc=0.354762
epoch=2 train_loss=1.375545 train_acc=0.309081 val_loss=1.277420 val_acc=0.380952
epoch=3 train_loss=1.366477 train_acc=0.307953 val_loss=1.305880 

## Combination 2 - unfreeze lr 1e-4 - result

In [16]:
show_run_result(LAST_RUN_DIR)


run_dir: /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s/videomae_cls_20260702_153544
## run_config.json True
run_id videomae_cls_20260702_153544
freeze_backbone False
epochs 10
batch_size 2
learning_rate 0.0001
weight_decay 0.0
best_epoch 2
best_val_accuracy 0.38095238095238093
train_rows 1773
val_rows 420
test_rows 212
## training_history.csv True
{'epoch': '1', 'train_loss': '1.388905', 'train_accuracy': '0.279188', 'val_loss': '1.265986', 'val_accuracy': '0.354762', 'test_loss': '1.288901', 'test_accuracy': '0.320755'}
{'epoch': '2', 'train_loss': '1.375545', 'train_accuracy': '0.309081', 'val_loss': '1.277420', 'val_accuracy': '0.380952', 'test_loss': '1.308387', 'test_accuracy': '0.353774'}
{'epoch': '3', 'train_loss': '1.366477', 'train_accuracy': '0.307953', 'val_loss': '1.305880', 'val_accuracy': '0.369048', 'test_loss': '1.323143', 'test_accuracy': '0.367925'}
{'epoch': '4', 'train_loss': '1.359884', 'train_accuracy': '0.324309', 'val_loss': 

## Combination 3 - regularized lr 5e-5 - define

In [17]:
EXPERIMENT = {'name': 'videomae_clip5s_exp3_lr5e-5_wd5e-2_e30', 'epochs': 30, 'batch_size': 2, 'learning_rate': 5e-05, 'weight_decay': 0.05, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


{'name': 'videomae_clip5s_exp3_lr5e-5_wd5e-2_e30', 'epochs': 30, 'batch_size': 2, 'learning_rate': 5e-05, 'weight_decay': 0.05, 'frame_count': 16, 'freeze_backbone': False}


## Combination 3 - regularized lr 5e-5 - train

In [18]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


$ /usr/bin/python ai/vision/train_videomae_classifier.py --manifest /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv --root-dir /workspace/SKN27-FINAL-3Team --output-dir /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s --label-column coarse_label --frame-count 16 --epochs 30 --batch-size 2 --learning-rate 5e-05 --weight-decay 0.05 --early-stopping-patience 2 --seed 42 --device auto --num-workers 0 --no-show-progress
run_id: videomae_cls_20260702_183519
device: cuda
manifest: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_clip_manifest_5s.csv
labels: {'차대보행자': 0, '차대이륜차': 1, '차대자전거': 2, '차대차': 3}
rows: train=1773 val=420 test=212
epoch=1 train_loss=1.372509 train_acc=0.298928 val_loss=1.226904 val_acc=0.388095
epoch=2 train_loss=1.299142 train_acc=0.376763 val_loss=1.248995 val_acc=0.371429
epoch=3 train_loss=1.178432 train_acc=0.478849 val_loss=1.250016 

## Combination 3 - regularized lr 5e-5 - result

In [19]:
show_run_result(LAST_RUN_DIR)


run_dir: /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_classification_clip5s/videomae_cls_20260702_183519
## run_config.json True
run_id videomae_cls_20260702_183519
freeze_backbone False
epochs 30
batch_size 2
learning_rate 5e-05
weight_decay 0.05
best_epoch 5
best_val_accuracy 0.5523809523809524
train_rows 1773
val_rows 420
test_rows 212
## training_history.csv True
{'epoch': '1', 'train_loss': '1.372509', 'train_accuracy': '0.298928', 'val_loss': '1.226904', 'val_accuracy': '0.388095', 'test_loss': '1.245629', 'test_accuracy': '0.410377'}
{'epoch': '2', 'train_loss': '1.299142', 'train_accuracy': '0.376763', 'val_loss': '1.248995', 'val_accuracy': '0.371429', 'test_loss': '1.269409', 'test_accuracy': '0.415094'}
{'epoch': '3', 'train_loss': '1.178432', 'train_accuracy': '0.478849', 'val_loss': '1.250016', 'val_accuracy': '0.409524', 'test_loss': '1.275564', 'test_accuracy': '0.396226'}
{'epoch': '4', 'train_loss': '0.959205', 'train_accuracy': '0.596729', 'val_loss': '

## Output review and next experiments

Current best VideoMAE result is Combination 3: `lr=5e-5`, `weight_decay=0.05`, unfreeze, best validation accuracy about `0.552` and test accuracy about `0.604` at epoch 5. After that, train accuracy rises while validation/test do not, so the next experiments should reduce overfitting instead of increasing epochs aggressively.

Also check the 5-second clip manifest first, because clip generation currently reduces `???` much more than the other labels. If the manifest is imbalanced, model tuning alone will not fix the result.


In [ ]:
from collections import Counter

for manifest_path in [DOWNLOAD_MANIFEST, CLIP_MANIFEST]:
    print(chr(10) + "##", manifest_path.name)
    with manifest_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    print("rows:", len(rows))
    for column in ["coarse_label", "split", "clip_status", "file_exists"]:
        if rows and column in rows[0]:
            print(column, dict(Counter(row.get(column) for row in rows)))


## Combination 4 - lower lr with same regularization

Keep the best direction from Combination 3, but lower the learning rate to reduce overfitting after epoch 5.


In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_exp4_lr3e-5_wd5e-2_e20', 'epochs': 20, 'batch_size': 2, 'learning_rate': 3e-05, 'weight_decay': 0.05, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 4 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 4 - result

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 5 - stronger regularization

Use the same learning rate as the current best run, but increase weight decay. This tests whether the overfitting after epoch 5 is mainly regularization-related.


In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_exp5_lr5e-5_wd1e-1_e20', 'epochs': 20, 'batch_size': 2, 'learning_rate': 5e-05, 'weight_decay': 0.1, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 5 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 5 - result

In [ ]:
show_run_result(LAST_RUN_DIR)
